In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind

pd.set_option("display.max_columns", None)

participants = pd.read_csv("participants.csv")

pre = pd.read_csv("pre_survey_submissions.csv")

post = pd.read_csv("post_survey_submissions.csv")

print("Participants:", participants.shape)
print("Pre Survey:", pre.shape)
print("Post Survey:", post.shape)

participants_features = participants[
    [
        "participant_id",
        "group"
    ]
]

pre_features = pre[
    [
        "participant_id",
        "score_correct",
        "score_total"
    ]
].rename(
    columns={
        "score_correct": "pre_score",
        "score_total": "pre_total"
    }
)

post_features = post[
    [
        "participant_id",
        "score_correct",
        "score_total"
    ]
].rename(
    columns={
        "score_correct": "post_score",
        "score_total": "post_total"
    }
)

# ============================================================
# MERGE DATASETS
# ============================================================

df = participants_features.merge(
    pre_features,
    on="participant_id",
    how="inner"
)

df = df.merge(
    post_features,
    on="participant_id",
    how="inner"
)

print("\nMerged Dataset Shape:", df.shape)

# ============================================================
# CREATE CUSTOM FEATURES
# ============================================================

df["pre_percentage"] = (
    df["pre_score"] / df["pre_total"]
) * 100

df["post_percentage"] = (
    df["post_score"] / df["post_total"]
) * 100

df["improvement"] = (
    df["post_percentage"] - df["pre_percentage"]
)

features = df[
    [
        "participant_id",
        "group",
        "pre_percentage",
        "post_percentage",
        "improvement"
    ]
]

print("\n==============================")
print("CUSTOM FEATURE DATASET")
print("==============================\n")

print(features.head())

print("\n==============================")
print("DESCRIPTIVE STATISTICS")
print("==============================\n")

descriptive_stats = features.groupby("group")[
    [
        "pre_percentage",
        "post_percentage",
        "improvement"
    ]
].agg(
    ["mean", "median", "std", "min", "max", "count"]
)

print(descriptive_stats.round(2))

print("\n==============================")
print("PARTICIPANT COUNTS")
print("==============================\n")

group_counts = features["group"].value_counts()

print(group_counts)

group_counts.plot(kind="bar")

plt.title("Participants per Group")
plt.xlabel("Group")
plt.ylabel("Number of Participants")

plt.xticks(rotation=0)

plt.show()

mean_scores = features.groupby("group")[
    [
        "pre_percentage",
        "post_percentage"
    ]
].mean()

print("\n==============================")
print("MEAN PRE AND POST SCORES")
print("==============================\n")

print(mean_scores.round(2))

mean_scores.plot(kind="bar")

plt.title("Average Pre-Test and Post-Test Scores")
plt.xlabel("Group")
plt.ylabel("Average Score Percentage")

plt.xticks(rotation=0)

plt.show()

print("\n==============================")
print("IMPROVEMENT ANALYSIS")
print("==============================\n")

improvement_stats = features.groupby("group")[
    "improvement"
].describe()

print(improvement_stats.round(2))



features.boxplot(
    column="improvement",
    by="group"
)

plt.title("Improvement Distribution by Group")
plt.suptitle("")

plt.xlabel("Group")
plt.ylabel("Improvement Percentage")

plt.show()

# ============================================================
# HISTOGRAM OF IMPROVEMENT
# ============================================================

control = features[
    features["group"] == "control"
]["improvement"]

test = features[
    features["group"] == "test"
]["improvement"]

plt.hist(control, alpha=0.7, label="Control")
plt.hist(test, alpha=0.7, label="Test")

plt.title("Distribution of Improvement")
plt.xlabel("Improvement Percentage")
plt.ylabel("Frequency")

plt.legend()

plt.show() 

print("\n==============================")
print("INDEPENDENT SAMPLES T-TEST")
print("==============================\n")

t_stat, p_value = ttest_ind(
    test,
    control,
    equal_var=False
)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

alpha = 0.05

if p_value < alpha:
    print("\nResult:")
    print("Reject the null hypothesis.")
    print("The test group performed significantly better.")
else:
    print("\nResult:")
    print("Fail to reject the null hypothesis.")
    print("No statistically significant difference detected.")

print("\n==============================")
print("EFFECT SIZE")
print("==============================\n")

mean_test = np.mean(test)
mean_control = np.mean(control)

std_test = np.std(test, ddof=1)
std_control = np.std(control, ddof=1)

n_test = len(test)
n_control = len(control)

pooled_std = np.sqrt(
    (
        ((n_test - 1) * std_test**2) +
        ((n_control - 1) * std_control**2)
    ) / (n_test + n_control - 2)
)

cohens_d = (
    mean_test - mean_control
) / pooled_std

print(f"Cohen's d: {cohens_d:.4f}")

if abs(cohens_d) < 0.2:
    interpretation = "Very Small Effect"
elif abs(cohens_d) < 0.5:
    interpretation = "Small Effect"
elif abs(cohens_d) < 0.8:
    interpretation = "Medium Effect"
else:
    interpretation = "Large Effect"

print("Interpretation:", interpretation)

summary = features.groupby("group").agg(
    participants=("participant_id", "count"),
    mean_pre_score=("pre_percentage", "mean"),
    mean_post_score=("post_percentage", "mean"),
    mean_improvement=("improvement", "mean"),
    std_improvement=("improvement", "std")
)

print("\n==============================")
print("SUMMARY TABLE")
print("==============================\n")

print(summary.round(2))

print("\n==============================")
print("DROPOUT RATE ANALYSIS")
print("==============================\n")

initial_counts = participants["group"].value_counts()

final_counts = features["group"].value_counts()

dropout_table = pd.DataFrame({
    "initial_participants": initial_counts,
    "completed_participants": final_counts
}).fillna(0)

dropout_table["dropouts"] = (
    dropout_table["initial_participants"]
    - dropout_table["completed_participants"]
)

dropout_table["dropout_rate_percent"] = (
    dropout_table["dropouts"]
    / dropout_table["initial_participants"]
) * 100

print(dropout_table.round(2))